# Donut fine-tune for InBody extraction — Colab runner

Runs `inform.training.train` on Colab GPU instead of the local 8GB laptop GPU (which is compute-bound on `donut-base`'s 2560x1920 canvas — ~150s/step locally).

**Before running**: Runtime -> Change runtime type -> pick a GPU (T4 is free tier; A100 needs Colab Pro and is much faster).

The dataset cell auto-resumes: first run regenerates the 5,000-sheet synthetic dataset (~60-90 min, installs headless Chrome) and saves a zip to your Drive. Any later run (e.g. after a disconnect) finds that zip and skips straight to unzipping it — seconds instead of an hour.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "feat/ocr-inbody-extraction"

# Private repo: add a fine-grained GitHub token (Contents: Read-only, scoped
# to QeekOw/InForm) as a Colab secret named GH_TOKEN (key icon in the left
# sidebar), then run this cell.
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
    GITHUB_REPO = f"https://{GH_TOKEN}@github.com/QeekOw/InForm.git"
except Exception:
    GITHUB_REPO = "https://github.com/QeekOw/InForm.git"  # public fallback

%cd /content
!rm -rf /content/repo
!git clone --branch $BRANCH --single-branch $GITHUB_REPO repo
%cd /content/repo
!pip install -q -e ".[training]"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
from pathlib import Path

DATA_ZIP = "/content/drive/MyDrive/inform/synthetic.zip"
DATA_DIR = "/content/data/synthetic"

if os.path.exists(DATA_ZIP):
    # Fast path: a prior run already generated + saved this dataset to Drive.
    print("Found saved dataset zip on Drive, skipping regeneration.")
    os.makedirs(DATA_DIR, exist_ok=True)
    shutil.unpack_archive(DATA_ZIP, DATA_DIR)
else:
    # First run, ~60-90 min. apt's chromium-browser on Ubuntu 22.04 (Colab's
    # OS) is a snap stub that doesn't work in a container (no snapd) — install
    # real Google Chrome instead, which _find_browser() looks for.
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    !apt-get -qq install -y ./google-chrome-stable_current_amd64.deb

    # Run generation in-kernel with the Python variable directly — do NOT go
    # through `!python -c "...$DATA_DIR..."`, whose shell interpolation can
    # come back empty and dump 10,000 files into the current directory.
    from inform.training.dataset import generate_dataset
    generate_dataset(Path(DATA_DIR))

    # Only cache to Drive if the full 10,000 files (5,000 sheets x png+json)
    # exist, so a partial/interrupted run doesn't get wrongly reused.
    n_files = len(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else 0
    if n_files >= 10000:
        os.makedirs(os.path.dirname(DATA_ZIP), exist_ok=True)
        shutil.make_archive(DATA_ZIP[:-4], "zip", DATA_DIR)
        print("Saved dataset zip to Drive for future resumes.")
    else:
        print(f"Only {n_files} files in {DATA_DIR} (expected 10000) — NOT saving to Drive. Re-run this cell.")

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/inform/checkpoints/donut-inbody"  # persists past session timeout
!mkdir -p "$CHECKPOINT_DIR"

In [ ]:
# batch-size 1 fits donut-base's 2560x1920 canvas on a T4 (16GB) with fp16 +
# gradient checkpointing (batch 2 OOMs by a hair). grad-accum 4 keeps the
# effective batch size at 4. A100 (40GB): raise batch to 4-8, drop grad-accum.
!python -m inform.training.train \
  --data-dir "$DATA_DIR" --output-dir "$CHECKPOINT_DIR" \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 --learning-rate 3e-5

If disconnected mid-run: re-run this notebook from the top, then re-run the training cell with `--resume` appended — checkpoints save every epoch to Drive, so nothing before the last completed epoch is lost.